In [1]:
import os
from os import system
from os.path import exists
from plistlib import dumps

from IPython.utils.capture import capture_output
from certifi import contents
from dotenv import load_dotenv
import json
import re
import time
import requests
from openai import OpenAI
import subprocess

In [2]:
load_dotenv()

True

In [3]:
os.environ["OPENROUTER_API_KEY"] = os.getenv("OPENROUTER_API_KEY")

In [4]:
client = OpenAI(

     base_url= "https://openrouter.ai/api/v1" ,
    api_key=os.environ["OPENROUTER_API_KEY"]
)

FREE_MODEL = "deepseek/deepseek-chat-v3.1"
TOOL_MODEL = "deepseek/deepseek-chat-v3.1"

In [5]:

NATIVE_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "It can read the file and return its output",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "Path of the file you want to read"
                    }
                },
                "required": ["path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "It can write to a file",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "Path of the file you want to write"
                    },
                    "content": {
                        "type": "string",
                        "description": "Content you want to write in the file"
                    }
                },
                "required": ["path", "content"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "It can return all the files in the directory",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "Directory path to list"
                    }
                },
                "required": ["path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "run_python",
            "description": "It can run Python code",
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {
                        "type": "string",
                        "description": "Code you want to run"
                    }
                },
                "required": ["code"]
            }
        }
    }
]


In [6]:
def read_file(path) :
    try :
        with open(path , "r") as f :
            content = f.read()
        return json.dumps({"path" : path , "content" : content[:3000] , "truncated" : len(content) > 3000})
    except Exception as e :
        return json.dumps({"error" : str(e)})

def write_file(path, content: str):
    try:
        d = os.path.dirname(path)

        if d:
            os.makedirs(d, exist_ok=True)

        with open(path, "w") as f:
            f.write(content)

        return json.dumps({
            "status": "success",
            "path": path,
            "bytes": len(content)
        })

    except Exception as e:
        return json.dumps({"error": str(e)})


import json

def run_python(code):
    try:
        result = subprocess.run(
            ["python3", "-c", code],
            capture_output=True,
            text=True,
            timeout=15,
        )

        return json.dumps({
            "stdout": result.stdout[:200],
            "stderr": result.stderr[:200],
            "exit_code": result.returncode,
        })

    except subprocess.TimeoutExpired:
        return json.dumps({"error": "Execution timed out after 15 seconds"})

    except Exception as e:
        return json.dumps({"error": str(e)})

def list_files(path=".") :
    try :
        items = sorted(os.listdir(path))
        return json.dumps({"path" : path , "files" : items[:50]})
    except Exception as e :
        return json.dumps({"error" : str(e)})

In [7]:
NATIVE_FNS = {
    "read_file" : read_file ,
    "write_file" : write_file ,
    "list_files" : list_files ,
    "run_python" : run_python
}

In [8]:

SYSTEM_PROMPT = """
You are an expert Python coding agent.

Your job is to solve the user's programming task by creating,
modifying, and testing Python code using the available tools.

Available tools:
{tool_description}

## WORKFLOW

### Phase 1: Understand the task
1. Read the user's request carefully.
2. Identify the requirements, constraints, inputs, and expected outputs.
3. Identify any missing information that prevents implementation.

### Phase 2: Create a plan
1. Create a numbered checklist of the steps needed to complete the task.
2. Make each step specific, actionable, and verifiable.
3. Include implementation, testing, and final verification.
4. Review the plan to ensure no requirements have been missed.
5. Revise the plan if necessary before execution.

### Phase 3: Execute the plan
1. Follow the plan in order.
2. Perform ONE operation at a time using the available tools.
3. After completing an operation, verify its result.
4. Mark a checklist item complete only after verifying that it succeeded.
5. If an operation fails, inspect the error, diagnose the cause,
   and attempt an appropriate correction.
6. If the plan needs to change, update the checklist and explain
   the reason briefly.
7. Do not claim that an operation succeeded unless the tool result
   supports that conclusion.

### Phase 4: Test the implementation
1. Run the relevant code or tests when execution tools are available.
2. Create a test file appropriate for the task.
3. Include normal cases, edge cases, and invalid inputs where relevant.
4. Execute the tests and inspect the results.
5. Fix failures and rerun the relevant tests.
6. If execution is unavailable, provide the test file and clearly
   state that the tests could not be executed.

### Phase 5: Final verification
1. Review the original requirements against the completed work.
2. Check for missing functionality, obvious errors, and untested behavior.
3. Ensure the final response accurately describes what was completed.
4. Report remaining limitations or failed tests honestly.

## RULES
- Follow the plan, but update it when evidence shows a change is needed.
- Perform one operation at a time.
- Do not mark a step complete before verifying its result.
- Do not skip testing merely because the code looks correct.
- Do not invent tool outputs, test results, files, or execution success.
- Use only the available tools for operations they support.
- Keep progress updates concise and useful.
- Do not expose hidden internal reasoning. Provide concise explanations
  of decisions, actions, and verification results.

## FINAL RESPONSE
Provide:
1. A brief summary of the implementation.
2. The files created or modified.
3. The tests performed and their actual results.
4. Any remaining issues or limitations.
"""


In [9]:
def run_native_agent(user_query, max_iteration=20, verbose=True):

    tool_desc = "\n".join(
        f"- {t['function']['name']}: {t['function']['description']}"
        for t in NATIVE_TOOLS
    )

    system = SYSTEM_PROMPT.format(
        tool_description=tool_desc
    )

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_query},
    ]

    if verbose:
        print("=" * 60)
        print(f"🧑 USER: {user_query}")
        print("=" * 60)

    for i in range(max_iteration):

        if verbose:
            print(f"\n--- Iteration {i + 1}/{max_iteration} ---")

        response = client.chat.completions.create(
            model=TOOL_MODEL,
            messages=messages,
            tools=NATIVE_TOOLS,
            temperature=0,
            max_tokens=1500,
        )

        msg = response.choices[0].message
        finish = response.choices[0].finish_reason

        # Handle tool calls
        if msg.tool_calls:

            messages.append(msg)

            for tc in msg.tool_calls:
                fn_name = tc.function.name
                fn_args = {}

                try:
                    fn_args = (
                        json.loads(tc.function.arguments)
                        if tc.function.arguments
                        else {}
                    )

                    if not isinstance(fn_args, dict):
                        raise ValueError(
                            "Tool arguments must be a JSON object."
                        )

                    if fn_name not in NATIVE_FNS:
                        raise ValueError(
                            f"Unknown tool: {fn_name}"
                        )

                    # Execute the requested function
                    result = NATIVE_FNS[fn_name](**fn_args)

                    # Tool output must be a string
                    result_text = (
                        result
                        if isinstance(result, str)
                        else json.dumps(result, default=str)
                    )

                except Exception as e:
                    result_text = json.dumps({
                        "error": f"Tool {fn_name} failed: {str(e)}"
                    })

                if verbose:
                    print(
                        f"🔧 TOOL: {fn_name}"
                        f"({json.dumps(fn_args)[:150]})"
                    )
                    print(f"👁️ RESULT: {result_text[:300]}")

                # Send the tool result back to the model
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result_text,
                })

        # Final answer
        elif finish == "stop" and msg.content:

            if verbose:
                print(f"\n✅ FINAL ANSWER:\n{msg.content[:1000]}")

            return {
                "answer": msg.content,
                "iteration": i + 1,
            }

        # Continue if the model returns text without finishing
        elif msg.content:

            messages.append({
                "role": "assistant",
                "content": msg.content,
            })

        else:
            if verbose:
                print(
                    "⚠️ Model stopped unexpectedly. "
                    f"Finish reason: {finish}"
                )
            break

    if verbose:
        print(
            f"\n⚠️ Maximum iterations ({max_iteration}) "
            "reached or the model stopped."
        )

    return {
        "answer": (
            "Maximum iterations reached or the model "
            "stopped unexpectedly."
        ),
        "iteration": max_iteration,
    }


In [10]:
result = run_native_agent(
    "Write a code to test if a number is odd or even"
)

🧑 USER: Write a code to test if a number is odd or even

--- Iteration 1/20 ---
🔧 TOOL: write_file({"path": "odd_even_test.py", "content": "def is_even(number):\n    \"\"\"\n    Check if a number is even.\n    Returns True if even, False if odd.\n  )
👁️ RESULT: {"status": "success", "path": "odd_even_test.py", "bytes": 741}

--- Iteration 2/20 ---
🔧 TOOL: run_python({"code": "python odd_even_test.py"})
👁️ RESULT: {"stdout": "", "stderr": "  File \u001b\"<string>\"\u001b, line \u001b1\u001b\n    python \u001bodd_even_test\u001b.py\n           \u001b^^^^^^^^^^^^^\u001b\n\u001bSyntaxError\u001b: \u001binvalid syntax\u001b\n", "exit_code": 1}

--- Iteration 3/20 ---
🔧 TOOL: read_file({"path": "odd_even_test.py"})
👁️ RESULT: {"path": "odd_even_test.py", "content": "def is_even(number):\n    \"\"\"\n    Check if a number is even.\n    Returns True if even, False if odd.\n    \"\"\"\n    return number % 2 == 0\n\ndef is_odd(number):\n    \"\"\"\n    Check if a number is odd.\n    Returns True

{'answer': '## Summary\n\nI have successfully created Python code to test if a number is odd or even. Here\'s what was implemented:\n\n### Files Created:\n1. **`odd_even_test.py`** - Main implementation with functions `is_even()` and `is_odd()`\n2. **`comprehensive_odd_even_test.py`** - Extended version with more test cases\n3. **`simple_odd_even.py`** - Minimalist version using lambda functions\n\n### Implementation Details:\n- **`is_even(number)`**: Returns `True` if number is even, `False` if odd\n- **`is_odd(number)`**: Returns `True` if number is odd, `False` if even\n- Both functions use the modulo operator (`%`) to check divisibility by 2\n- Also demonstrated bitwise AND method (`num & 1 == 0`) as an alternative\n\n### Tests Performed:\nThe code was tested with various numbers including:\n- Positive integers (0, 1, 2, 3, 4, 5, 10, 15, 100, 101)\n- Negative integers (-1, -2, -3, -4, -5)\n- Large numbers (999999999, -999999999)\n- Edge cases (2147483647, -2147483648)\n\n### Result